# Лабораторна робота № 5
## Географічні дані OpenStreetMap: від тегів до маршруту

**Інтелектуальний аналіз та візуалізація даних** · 4 години · 4 бали

---

### Постановка

Ви — аналітик у туристичному застосунку. Завдання: для заданого міста
**відібрати десять найвартісніших історико-культурних об'єктів** і
**спланувати піший маршрут**, який починається на залізничному вокзалі,
обходить усі десять і повертається на вокзал.

Слово «найвартісніші» в даних не записане. Критерій ви побудуєте самі —
і захистите його.

Потім те саме для **тридцяти** об'єктів. Результат цієї, більшої задачі
стане базою порівняння в **лабораторній роботі № 8**, де ту саму задачу
розв'язуватимуть генетичним та мурашиним алгоритмами.

### Що здаєте

1. Цей ноутбук із виконаними комірками (усі виводи збережені).
2. Файл `lr5_результати.json` — результати для ЛР8.
3. Карту `lr5_маршрут.html`.
4. Звіт у кінці ноутбука: відповіді на сім питань.

### Що оцінюється

| критерій | 0 | 2 | 3 | 4 |
|---|---|---|---|---|
| Відбір об'єктів | теги взято навмання | фільтр працює | фільтр обґрунтовано | показано, що відкинуто і чому |
| Критерій рейтингу | немає | є формула | ваги пояснено | порівняно з альтернативним критерієм |
| Маршрут | не будується | будується | перевірено коректність | пояснено різницю методів числом |
| Карта | немає | точки на карті | маршрут по ребрах графа | видно різницю з прямими відрізками |
| Звіт | немає | формально | з числами | з висновком, якого не було в завданні |

---
## 0. Ваш варіант

Впишіть прізвище та номер у списку групи. Місто підтягнеться само.

In [1]:
STUDENT_NAME = 'Смачило Іванна'
LIST_NUMBER  = 11

CITIES = [
    'Львів', 'Київ', 'Харків', 'Одеса', 'Дніпро',
    'Чернівці', 'Івано-Франківськ', 'Тернопіль', 'Ужгород', 'Луцьк',
    'Рівне', 'Хмельницький', 'Вінниця', 'Житомир', 'Чернігів',
    'Черкаси', 'Полтава', 'Суми', 'Кропивницький', 'Запоріжжя',
    'Миколаїв', 'Кривий Ріг', 'Кам\u2019янець-Подільський', 'Мукачево',
    'Коломия', 'Дрогобич', 'Біла Церква', 'Ніжин', 'Остріг', 'Умань',
]

assert STUDENT_NAME and LIST_NUMBER > 0, 'впишіть прізвище та номер у списку'
CITY = CITIES[(LIST_NUMBER - 1) % len(CITIES)] + ', Україна'
print('варіант %d: %s' % (LIST_NUMBER, CITY))

варіант 11: Рівне, Україна


> Якщо викладач призначив вам інше місто — просто перепишіть `CITY` вручну
> і зазначте це у звіті.

In [2]:
import importlib.util, subprocess, sys
REQUIRED = ['osmnx', 'geopandas', 'shapely', 'pyproj', 'pyarrow',
            'folium', 'mapclassify', 'h3', 'networkx']
missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    print('встановлюю:', ', '.join(missing), '...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('усі пакети на місці')

усі пакети на місці


In [3]:
import warnings; warnings.filterwarnings('ignore')
import json, time, os, collections
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
import folium
from xyzservices import providers as xyz

BASEMAP = xyz.Esri.WorldGrayCanvas   # CARTO з 2026 р. вимагає ключ API
pd.set_option('display.width', 200)
print('geopandas', gpd.__version__)

geopandas 1.2.0


---
## 1. Два способи дістати дані

Робоча програма вимагає порівняти два способи доступу **за обсягом переданих
даних і за часом**:

* **живий запит** — `osmnx` питає Overpass API просто зараз;
* **знімок** — ті самі дані, збережені у файли GeoParquet, читаються з диска.

Нижче — живий запит. Запустіть його **один раз**: він збереже теку зі знімком,
і всі наступні запуски читатимуть з неї.

> Overpass — безкоштовний сервіс і обмежує частоту. Якщо запит не пройшов —
> зачекайте хвилину й повторіть. Якщо не проходить взагалі, візьміть
> резервний знімок у викладача і вкажіть це у звіті.

In [4]:
import osmnx as ox

OUT = 'osm_' + CITY.split(',')[0].strip().lower().replace(' ', '_')
os.makedirs(OUT, exist_ok=True)
ox.settings.use_cache = True
ox.settings.cache_folder = os.path.join(OUT, 'cache')
ox.settings.requests_timeout = 300
ox.settings.overpass_rate_limit = True

TOURISM_OK = ['museum', 'attraction', 'gallery', 'artwork', 'viewpoint']

def cache_bytes():
    import pathlib
    p = pathlib.Path(ox.settings.cache_folder)
    return sum(f.stat().st_size for f in p.glob('*.json')) if p.exists() else 0

print('тека знімка:', OUT)

тека знімка: osm_рівне


### Дві налаштовані дрібниці, без яких запит не пройде

Це не формальності — кожна з них ламає збір по-своєму, і жодна не схожа
на свою справжню причину.

**Представитись.** Overpass відповідає **HTTP 406** клієнтові, який прийшов
із типовим заголовком `python-requests`. `osmnx` надсилає свій `User-Agent`
сам, але якщо ви звертаєтесь до сервісу напряму — надсилайте теж.
Рядок має бути **лише латиницею**: HTTP-заголовки передаються в latin-1,
і кирилиця кладе запит помилкою `UnicodeEncodeError` ще до відправлення.

**Обрати машину.** За іменем `overpass-api.de` стоїть кілька серверів, і
частина з них відмовляє у з’єднанні. `osmnx` закріплює одну адресу на
всю сесію — навмисно, щоб не порушувати облік слотів. Якщо закріпиться та,
що відмовляє, впаде весь збір. Тому адресу обираємо самі, вже перевірену.

In [ ]:
import requests

USER_AGENT = ('LvivPolytechnic-SK26-coursework/1.0 '
              '(https://lpnu.ua; student lab assignment)')
assert USER_AGENT.isascii(), 'HTTP-заголовки — latin-1, кирилиця тут не працює'
ox.settings.http_user_agent = USER_AGENT

MIRRORS = ['https://overpass-api.de/api',
           'https://overpass.private.coffee/api',
           'https://z.overpass-api.de/api']
PROBE = '[out:json][timeout:10];node(49.8419,24.0315,49.8420,24.0316);out count;'

def pick_mirror():
    busy = None
    for base in MIRRORS:
        try:
            r = requests.post(base + '/interpreter', data={'data': PROBE},
                              headers={'User-Agent': USER_AGENT}, timeout=40)
        except requests.exceptions.ReadTimeout:
            print('  %-42s ReadTimeout (TLS працює, сервер повільний)' % base)
            busy = busy or base; continue
        except Exception as e:
            print('  %-42s %s' % (base, type(e).__name__)); continue
        print('  %-42s HTTP %s' % (base, r.status_code))
        if r.status_code == 200 and r.text.lstrip().startswith('{'):
            return base
        if r.status_code in (429, 504) and busy is None:
            busy = base
    return busy

chosen = pick_mirror()
assert chosen, 'жодне дзеркало Overpass не відповідає з цієї мережі: спробуйте через хвилину або з іншої мережі'
ox.settings.overpass_url = chosen
print('обрано дзеркало:', chosen)

  https://overpass-api.de/api                HTTP 200
обрано дзеркало: https://overpass-api.de/api


> Якщо жодна машина не відповіла — виконайте комірку ще раз за хвилину,
> або візьміть у викладача резервний знімок і зазначте це у звіті.
> Інші дзеркала перелічені на сторінці
> [Overpass API](https://wiki.openstreetmap.org/wiki/Overpass_API);
> адресу можна підставити у `ox.settings.overpass_url`.

### Завдання 1.1

Допишіть функцію `fetch`: вона має зберегти шар у `.parquet`, а якщо файл
уже є — прочитати його, **не звертаючись до мережі**. Поверніть також,
скільки МБ пішло мережею і скільки секунд це зайняло.

Підказка щодо `clean`: у OSM одна колонка може містити і число, і список
(ребро, склеєне з кількох шляхів). Parquet вимагає один тип на колонку,
тому неоднорідну колонку треба звести до тексту **цілком**.

In [ ]:
def clean(gdf):
    g = gdf.reset_index()
    for c in g.columns:
        if c == 'geometry':
            continue
        boxed = g[c].map(lambda v: isinstance(v, (list, dict, set, tuple))).any()
        if boxed or g[c].dtype == object:
            g[c] = g[c].map(
                lambda v: None if v is None or (isinstance(v, float) and pd.isna(v))
                else (json.dumps(v, ensure_ascii=False)
                      if isinstance(v, (list, dict, set, tuple)) else str(v))
            ).astype('string')
    return g


def with_retry(call, tries=6, wait=30):
    """Overpass часто повертає 502/429/504 текстом (усі воркери зайняті). Повторюємо з наростаючою паузою."""
    for k in range(1, tries + 1):
        try:
            return call()
        except Exception as e:
            if k == tries:
                raise
            print('   спроба %d з %d не вдалась (%s); чекаю %d с' % (k, tries, type(e).__name__, wait))
            time.sleep(wait); wait *= 2


COST = {}

def fetch(name, getter):
    path = os.path.join(OUT, name + '.parquet')
    if os.path.exists(path) and os.path.getsize(path) > 0:
        t0 = time.time()
        g = gpd.read_parquet(path)
        COST[name] = {'спосіб': 'знімок', 'записів': len(g),
                      'передано_МБ': 0.0,
                      'файл_МБ': round(os.path.getsize(path) / 1e6, 3),
                      'секунд': round(time.time() - t0, 3)}
        return g

    t0, b0 = time.time(), cache_bytes()            
    g = clean(with_retry(getter))                            
    g.to_parquet(path, index=False)
    sec = time.time() - t0
    mb = (cache_bytes() - b0) / 1e6                
    COST[name] = {'спосіб': 'живий запит', 'записів': len(g),
                  'передано_МБ': round(mb, 3),
                  'файл_МБ': round(os.path.getsize(path) / 1e6, 3),
                  'секунд': round(sec, 1)}
    
    mp = os.path.join(OUT, 'manifest.json')
    with open(mp, 'w', encoding='utf-8') as f:
        json.dump(COST, f, ensure_ascii=False, indent=2)

    return gpd.read_parquet(path)

In [7]:
boundary = fetch('boundary', lambda: ox.geocode_to_gdf(CITY))
poly = boundary.geometry.iloc[0]
print('межа міста отримана, площа %.1f км²'
      % (boundary.to_crs(32635).area.iloc[0] / 1e6))

межа міста отримана, площа 58.2 км²


In [ ]:
heritage = fetch('heritage', lambda: ox.features_from_polygon(
    poly, {'historic': True, 'tourism': TOURISM_OK}))
stations = fetch('stations', lambda: ox.features_from_polygon(
    poly, {'railway': ['station', 'halt'], 'building': ['train_station']}))
amenities = fetch('amenities', lambda: ox.features_from_polygon(
    poly, {'amenity': ['cafe', 'restaurant', 'drinking_water', 'toilets']}))

NODES_P = os.path.join(OUT, 'road_nodes.parquet')
EDGES_P = os.path.join(OUT, 'road_edges.parquet')

if os.path.exists(NODES_P) and os.path.exists(EDGES_P):
    t0 = time.time()
    nodes, edges = gpd.read_parquet(NODES_P), gpd.read_parquet(EDGES_P)
    COST['мережа walk'] = {'спосіб': 'знімок', 'записів': len(nodes) + len(edges),
                           'передано_МБ': 0.0,
                           'файл_МБ': round((os.path.getsize(NODES_P)
                                             + os.path.getsize(EDGES_P)) / 1e6, 3),
                           'секунд': round(time.time() - t0, 3)}
else:
    t0, b0 = time.time(), cache_bytes()
    G_raw = with_retry(lambda: ox.graph_from_polygon(poly, network_type='walk'))
    nodes, edges = ox.convert.graph_to_gdfs(G_raw)
    clean(nodes).to_parquet(NODES_P, index=False)
    clean(edges).to_parquet(EDGES_P, index=False)
    nodes, edges = gpd.read_parquet(NODES_P), gpd.read_parquet(EDGES_P)
    COST['мережа walk'] = {'спосіб': 'живий запит', 'записів': len(nodes) + len(edges),
                           'передано_МБ': round((cache_bytes() - b0) / 1e6, 3),
                           'файл_МБ': round((os.path.getsize(NODES_P)
                                             + os.path.getsize(EDGES_P)) / 1e6, 3),
                           'секунд': round(time.time() - t0, 1)}

print('вузлів %d | ребер %d' % (len(nodes), len(edges)))

mp = os.path.join(OUT, 'manifest.json')
with open(mp, 'w', encoding='utf-8') as f:
    json.dump(COST, f, ensure_ascii=False, indent=2)

вузлів 9866 | ребер 27004


### Завдання 1.2

Зведіть `COST` у таблицю і дайте відповідь у звіті: **у скільки разів
читання знімка швидше за живий запит** і **скільки мегабайтів ви не передали
мережею** при повторному запуску.

Контрольна ознака правильності: при **другому** запуску ноутбука всі рядки
мають мати спосіб «знімок», а `передано_МБ` — нуль.

In [9]:
display(pd.DataFrame(COST).T)

mp = os.path.join(OUT, 'manifest.json')
if os.path.exists(mp):
    m = json.load(open(mp, encoding='utf-8'))
    live = {k: v for k, v in m.items() if not k.startswith('_') and isinstance(v, dict) and 'секунд' in v}
    t_live = sum(v['секунд'] for v in live.values()); mb_live = sum(v['передано_МБ'] for v in live.values())
    t_snap = sum(v['секунд'] for v in COST.values())
    print('\nживий запит (з manifest.json): %.1f с, %.2f МБ' % (t_live, mb_live))
    print('знімок (цей запуск)           : %.3f с, 0 МБ' % t_snap)
    if t_snap > 0:
        print('знімок швидший у %.0f раз(ів), не передано %.2f МБ' % (t_live / t_snap, mb_live))

,спосіб,записів,передано_МБ,файл_МБ,секунд
boundary,знімок,1,0.0,0.028,0.108
heritage,знімок,119,0.0,0.061,0.012
stations,знімок,5,0.0,0.032,0.011
amenities,знімок,235,0.0,0.057,0.012
мережа walk,знімок,36870,0.0,1.853,0.026


> **Чому пішохідна мережа, а не автомобільна.** У старому місті музеї стоять
> за сотню метрів один від одного, а автомобільних вузлів там одиниці. Кілька
> пам'яток сядуть на **один** вузол графа, і матриця віддалей скаже, що між
> ними нуль метрів. Задача розв'яжеться, відповідь буде неправильною, і
> повідомлення про помилку не з'явиться. У кроці 4 ви це перевірите явно.

---
## 2. Хто претендує на звання пам'ятки

У OpenStreetMap немає тега «визначна пам'ятка». Є два сімейства:

* `historic=*` — `castle`, `church`, `monument`, `memorial`, `ruins`, `tower`, `yes`…
* `tourism=*` — `museum`, `gallery`, `attraction`, `artwork`, `viewpoint`

Перетин неповний: музей у сучасній будівлі має лише `tourism`, руїни фортеці —
лише `historic`. Тому беремо **об'єднання**.

📖 <https://wiki.openstreetmap.org/wiki/Map_Features>

### Завдання 2.1

Відберіть кандидатів і **назвіть у звіті, скільки об'єктів ви відкинули й чому**.

In [10]:
MINOR = ['boundary_stone', 'wayside_cross', 'wayside_shrine', 'milestone', 'survey_point']
for col in ['historic', 'tourism', 'name', 'wikidata', 'wikipedia']:
    if col not in heritage.columns:
        heritage[col] = None

has_tag  = heritage['tourism'].isin(TOURISM_OK) | heritage['historic'].notna()
has_name = heritage['name'].notna()
is_minor = heritage['historic'].isin(MINOR)

cand = heritage[has_tag & has_name & ~is_minor].reset_index(drop=True)
cand['key'] = cand['element'].astype(str) + cand['id'].astype(str)

print('усього у шарі heritage :', len(heritage))
print('підходять за тегами    :', int(has_tag.sum()))
print('з них без name         :', int((has_tag & ~has_name).sum()))
print('з них дрібні знаки     :', int((has_tag & has_name & is_minor).sum()))
print('лишилось кандидатів    :', len(cand))
print('відкинуто всього       :', len(heritage) - len(cand))

усього у шарі heritage : 119
підходять за тегами    : 119
з них без name         : 14
з них дрібні знаки     : 0
лишилось кандидатів    : 105
відкинуто всього       : 14


In [11]:
assert 'cand' in dir(), 'спершу виконайте завдання 2.1'
print('=== historic ===');  print(cand['historic'].value_counts().head(10).to_string())
print()
print('=== tourism ===');   print(cand['tourism'].value_counts().head(10).to_string())
print()
print('типи геометрії:', cand.geometry.geom_type.value_counts().to_dict())

=== historic ===
historic
memorial               49
monument                5
tank                    3
cannon                  3
archaeological_site     2
vehicle                 2
building                1
aircraft                1
city_gate               1

=== tourism ===
tourism
artwork       24
attraction     9
museum         3
gallery        2
viewpoint      1

типи геометрії: {'Point': 101, 'Polygon': 4}


> **Перевірте себе.** Серед типів геометрії мають бути і `Point`, і `Polygon`.
> Якщо самі лише полігони — ви відфільтрували зайве: пам'ятники, меморіали та
> оглядові точки в OSM є **вузлами**, у них немає будівлі.

### Якщо кандидатів менше тридцяти

Невелике місто може дати 15–20 об'єктів. Тоді:

1. розширте перелік `tourism` (наприклад, додайте `theatre`, `place_of_worship`);
2. або зменште `N_LARGE` до того, що є;

і **обов'язково зазначте це у звіті** разом із числом.

---
## 3. Критерій «найкращі»

Ключовий крок роботи. Стовпця «цінність» у даних немає. Його треба
сконструювати і захистити.

Сигнали, які є в OSM:

| сигнал | що означає |
|---|---|
| `wikidata` / `wikipedia` | про об'єкт є стаття — хтось сторонній визнав його вартим опису |
| кількість колонок `name:*` | назву переклали N мовами — перекладають те, що шукають іноземці |
| `tourism=museum/gallery` | призначений для відвідування |
| `historic` заповнений | має історичну природу |
| площа полігона | розмір — **найслабший сигнал**: великим буває і склад |

### Завдання 3.1

Побудуйте числовий рейтинг. Вимоги:

* використати **щонайменше три** різні сигнали;
* звести різнорідні величини до спільної шкали (підказка: `rank(pct=True)`);
* **пояснити у звіті, чому саме такі ваги**.

In [ ]:
metric = cand.to_crs(32635)
cand['area_m2'] = np.where(metric.geometry.geom_type.isin(['Polygon', 'MultiPolygon']),
                           metric.area.values, 0.0)

try:
    bl = fetch('buildings', lambda: ox.features_from_polygon(poly, {'building': True}))
    bl = bl.to_crs(32635)
    bl = bl[bl.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
    bl['b_area'] = bl.area
    pts_c = gpd.GeoDataFrame({'key': cand['key']},
                             geometry=metric.geometry.representative_point().values, crs=32635)
    joined = gpd.sjoin(pts_c, bl[['b_area', 'geometry']], how='left', predicate='within')
    area_b = joined.groupby('key')['b_area'].max()
    cand['area_m2'] = cand['key'].map(area_b).fillna(cand['area_m2'])
except Exception as e:
    print('шар будівель недоступний, площа лише для полігонів:', str(e)[:80])
print('об\'єктів із площею > 0:', int((cand['area_m2'] > 0).sum()), 'з', len(cand))

for col in ['wikidata', 'wikipedia']:
    if col not in cand.columns:
        cand[col] = None

lang_cols = [c for c in cand.columns if c.startswith('name:') and 'etymology' not in c and c != 'name:uk']

cand['wiki']     = (cand['wikidata'].notna() | cand['wikipedia'].notna()).astype(int)
cand['n_lang']   = cand[lang_cols].notna().sum(axis=1)
cand['is_visit'] = cand['tourism'].isin(['museum', 'gallery']).astype(int)
cand['is_hist']  = cand['historic'].notna().astype(int)

W = {'wiki': 0.35, 'n_lang': 0.30, 'is_visit': 0.15, 'is_hist': 0.10, 'area': 0.10}
cand['score'] = (W['wiki'] * cand['wiki']
               + W['n_lang'] * cand['n_lang'].rank(pct=True)
               + W['is_visit'] * cand['is_visit']
               + W['is_hist'] * cand['is_hist']
               + W['area'] * cand['area_m2'].rank(pct=True))

print('кандидатів   :', len(cand))
print('мають wiki   :', int(cand['wiki'].sum()))
print('мають 1+ мову:', int((cand['n_lang'] > 0).sum()), '| колонок name:* :', len(lang_cols))

об'єктів із площею > 0: 13 з 105
кандидатів   : 105
мають wiki   : 7
мають 1+ мову: 11 | колонок name:* : 3


### Завдання 3.2 — обов'язкове порівняння

Побудуйте **другий** рейтинг за найпростішим критерієм — площею будівлі —
і порахуйте, скільки об'єктів збігається в перших десяти.

Це головне питання роботи: обидва списки отримані з тих самих даних, тим
самим кодом, без жодної помилки. Якщо вони різні — поясніть у звіті, що це
означає для слова «найкращі».

In [13]:
N_SMALL, N_LARGE = 10, 30
N_LARGE = min(N_LARGE, len(cand))      # у малому місті кандидатів може бути менше за 30

# при рівному балі сортуємо за назвою, щоб порядок не залежав від комп'ютера
by_score = cand.sort_values(['score', 'name'], ascending=[False, True], kind='stable').reset_index(drop=True)
top_score = by_score.head(N_LARGE).reset_index(drop=True)
top10     = top_score.head(N_SMALL)
top_naive = (cand.sort_values(['area_m2', 'name'], ascending=[False, True], kind='stable')
                 .head(N_SMALL).reset_index(drop=True))
overlap   = len(set(top10['key']) & set(top_naive['key']))

print('N_LARGE =', N_LARGE)
print('перетин: %d з %d' % (overlap, N_SMALL))
display(pd.DataFrame({'за вашим критерієм': list(top10['name']),
                      'за площею': list(top_naive['name'])}))

N_LARGE = 30
перетин: 3 з 10


,за вашим критерієм,за площею
0,Загиблим у війні 1941-1945 рр.,Басівкутське городище
1,ЗіС-3,Меморіал Слави
2,Климу Савуру,Краєзнавчий музей
3,Марії Несвицькій (Рівненській),Меморіальна дошка В. Г. Короленку
4,Симону Петлюрі,Меморіальна дошка митлополиту Іларіону (Іван О...
5,Тарасу Шевченку,Євро-Арт
6,Таємничі підземелля Рівного,Євро-Арт
7,Літературний музей Уласа Самчука,"Мурал (Digital) ""Моє місто"""
8,Музей бурштину,Літературний музей Уласа Самчука
9,Краєзнавчий музей,Музей бурштину


### Завдання 3.3 — дублікати

OSM наповнюють різні люди. Той самий музей трапляється двічі під різними
назвами, а вежа буває окремим об'єктом усередині палацу. Знайдіть у `top_score`
пари, що стоять ближче **60 метрів** одна до одної, і вирішіть, що з ними робити.

Рішення (лишити / злити / видалити) опишіть у звіті.

In [14]:
def close_pairs(gdf, limit_m=60):
    p = gdf.to_crs(32635).geometry.representative_point()
    XY = np.c_[p.x.values, p.y.values]
    D = np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))
    np.fill_diagonal(D, np.inf)
    out = [(i, j, D[i, j]) for i in range(len(XY)) for j in range(i + 1, len(XY))
           if D[i, j] < limit_m]
    return sorted(out, key=lambda t: t[2])

# 1) пари ближче 60 м у top_score (числа — місце в рейтингу)
pairs = close_pairs(top_score)
print('пар ближче 60 м у top_score:', len(pairs))
for i, j, d in pairs:
    print('  %5.1f м | #%d %s  <->  #%d %s' % (d, i + 1, top_score['name'][i], j + 1, top_score['name'][j]))

# 2) рішення: з кожної групи близьких об'єктів лишаємо той, що вищий у рейтингу
def dedupe(df, limit_m=60):
    pts = df.to_crs(32635).geometry.representative_point()
    keep = []
    for i in range(len(df)):
        if all(pts.iloc[i].distance(pts.iloc[k]) >= limit_m for k in keep):
            keep.append(i)
    return df.iloc[keep].reset_index(drop=True)

overlap_raw = overlap
old_keys = set(top_score['key'])
cand_clean = dedupe(by_score)
top_score = cand_clean.head(N_LARGE).reset_index(drop=True)   # добираємо наступних, щоб знову було N_LARGE
top10     = top_score.head(N_SMALL)
overlap   = len(set(top10['key']) & set(top_naive['key']))

print()
print('видалено близьких дублікатів серед усіх кандидатів:', len(cand) - len(cand_clean))
print('змінилось у top_score                           :', len(set(top_score['key']) - old_keys))
print('перетин із площею: було %d, стало %d з %d' % (overlap_raw, overlap, N_SMALL))

пар ближче 60 м у top_score: 19
    3.5 м | #20 Меморіальна дошка В. Г. Короленку  <->  #21 Меморіальна дошка митлополиту Іларіону (Іван Огієнко)
    6.2 м | #8 Літературний музей Уласа Самчука  <->  #9 Музей бурштину
    8.0 м | #2 ЗіС-3  <->  #28 VW T4 (уламки)
   12.4 м | #17 Євро-Арт  <->  #18 Євро-Арт
   23.0 м | #10 Краєзнавчий музей  <->  #20 Меморіальна дошка В. Г. Короленку
   24.3 м | #6 Тарасу Шевченку  <->  #7 Таємничі підземелля Рівного
   26.0 м | #21 Меморіальна дошка митлополиту Іларіону (Іван Огієнко)  <->  #28 VW T4 (уламки)
   26.2 м | #10 Краєзнавчий музей  <->  #21 Меморіальна дошка митлополиту Іларіону (Іван Огієнко)
   29.4 м | #20 Меморіальна дошка В. Г. Короленку  <->  #28 VW T4 (уламки)
   29.9 м | #2 ЗіС-3  <->  #21 Меморіальна дошка митлополиту Іларіону (Іван Огієнко)
   33.1 м | #2 ЗіС-3  <->  #20 Меморіальна дошка В. Г. Короленку
   37.5 м | #13 Жертвам Чорнобильскої катастрофи  <->  #16 Пам'ятна дошка гетьману Івану Мазепі
   38.5 м | #9 Музей бурштину  <

---
## 4. Старт: вокзал, і прив'язка точок до графа

In [15]:
print('станції у шарі:', len(stations))
print(stations[['name', 'railway', 'building']].dropna(how='all').head(12).to_string())

станції у шарі: 5
                       name  railway       building
0                    Озерна  station            NaN
1              Партизанська  station            NaN
2                     Рівне  station            NaN
3                 Дворецька     halt            NaN
4  Залізничний вокзал Рівне      NaN  train_station


In [ ]:
city_name = CITY.split(',')[0].strip()
for col in ['railway', 'building', 'name']:
    if col not in stations.columns:
        stations[col] = None

st = stations[(stations['railway'].isin(['station', 'halt']) | (stations['building'] == 'train_station'))
              & stations['name'].notna()]
exact   = st[st['name'] == city_name]
by_name = st[st['name'].str.contains(city_name, case=False, regex=False)]
pool = exact if len(exact) else by_name if len(by_name) else st
pool = pool.assign(_rank=(pool['railway'] != 'station').astype(int)).sort_values('_rank')
station = pool.head(1).drop(columns='_rank').reset_index(drop=True)

print('старт:', station['name'].iloc[0])
print('тип геометрії:', station.geometry.geom_type.iloc[0], '| варіантів у пулі:', len(pool))

старт: Рівне
тип геометрії: Point | варіантів у пулі: 1


In [ ]:
G = nx.DiGraph()
for t in nodes.itertuples():
    G.add_node(t.osmid, x=t.x, y=t.y)
for t in edges.itertuples():
    L = float(t.length)
    if G.has_edge(t.u, t.v):
        G[t.u][t.v]['length'] = min(G[t.u][t.v]['length'], L)
    else:
        G.add_edge(t.u, t.v, length=L)
print('граф: %d вузлів, %d ребер' % (G.number_of_nodes(), G.number_of_edges()))

NODES_M = nodes.to_crs(32635)
NODE_XY = np.c_[NODES_M.geometry.x.values, NODES_M.geometry.y.values]
NODE_ID = nodes['osmid'].values

def snap(points_gdf):
    pmet = points_gdf.to_crs(32635)
    ids, dist = [], []
    for p in pmet.geometry:
        d2 = (NODE_XY[:, 0] - p.x) ** 2 + (NODE_XY[:, 1] - p.y) ** 2
        k = int(np.argmin(d2))
        ids.append(NODE_ID[k]); dist.append(float(np.sqrt(d2[k])))
    return np.array(ids), np.array(dist)

def make_points(top):
    names = ['СТАРТ: ' + str(station['name'].iloc[0])] + list(top['name'])
    geoms = [station.geometry.iloc[0].representative_point()] + \
            [g.representative_point() for g in top.geometry]
    return gpd.GeoDataFrame({'name': names}, geometry=geoms, crs='EPSG:4326')

граф: 9866 вузлів, 26747 ребер


### Завдання 4.1 — обов'язкова перевірка

Напишіть `check_snap`, яка повідомляє:

* скільки точок і скільки **різних** вузлів вони зайняли;
* медіану й максимум відстані «точка → вузол»;
* якщо вузлів менше, ніж точок — **назви точок, що злиплись**.

**Контрольна ознака:** на пішохідній мережі різних вузлів має бути **стільки ж,
скільки точок**. Якщо менше — ви взяли автомобільну мережу або місто надто мале;
зафіксуйте це у звіті, бо далі такі пари отримають нульову віддаль.

In [18]:
def check_snap(points, ids, dist, label):
    n, u = len(points), len(set(ids))
    print('%s: точок %d, різних вузлів %d' % (label, n, u))
    print('   відстань точка -> вузол: медіана %.1f м, максимум %.1f м'
          % (np.median(dist), np.max(dist)))
    if u < n:
        print('   УВАГА: точки злиплись на одному вузлі:')
        for node, cnt in collections.Counter(ids).items():
            if cnt > 1:
                names = [str(points['name'].iloc[i]) for i in range(n) if ids[i] == node]
                print('     вузол %s: %s' % (node, ' | '.join(names)))
    else:
        print('   усі точки на різних вузлах: добре')

---
## 5. Матриця віддалей

Дві матриці: **пряма лінія** і **дорогою**. Їхнє відношення — коефіцієнт
обходу — показує, наскільки пряма лінія бреше.

### Завдання 5.1

In [ ]:
def road_matrix(ids):
    k = len(ids)
    D = np.full((k, k), np.inf)
    for i, s in enumerate(ids):
        lengths = nx.single_source_dijkstra_path_length(G, s, weight='length')
        for j, t in enumerate(ids):
            D[i, j] = lengths.get(t, np.inf)
    np.fill_diagonal(D, 0.0)
    return D

def straight_matrix(points):
    p = points.to_crs(32635)
    XY = np.c_[p.geometry.x.values, p.geometry.y.values]
    return np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))

def detour_factor(D, E):
    ok = np.isfinite(D) & (E > 0)
    return float(np.median(D[ok] / E[ok]))

**Контрольні ознаки — звіртесь, не підганяючи:**

* коефіцієнт обходу для пішохідної мережі у місті зазвичай **1,2–1,6**;
* якщо у вас рівно **1,0** — ви порівнюєте матрицю саму із собою;
* якщо **більше 2** — найімовірніше взято автомобільну мережу з односторонніми
  вулицями, або місто має річку/залізницю, яку нема де перетнути. Друге —
  цікавий результат, перше — помилка. Розберіться, що саме у вас, і напишіть це.

Якщо серед віддалей трапились `inf` — частина графа не зв'язана з рештою.
Не замінюйте `inf` великим числом мовчки: спершу з'ясуйте, які це точки.

In [20]:
pts10 = make_points(top10)
ids10, dist10 = snap(pts10)
check_snap(pts10, ids10, dist10, 'мала задача')

t0 = time.time(); D10 = road_matrix(ids10); print('матриця за %.3f с' % (time.time() - t0))
E10 = straight_matrix(pts10)
print('недосяжних пар    :', int(np.isinf(D10).sum()))
print('асиметрія         : %.0f м' % float(np.max(np.abs(D10 - D10.T))))
print('коефіцієнт обходу : %.2f' % detour_factor(D10, E10))

мала задача: точок 11, різних вузлів 11
   відстань точка -> вузол: медіана 17.0 м, максимум 30.4 м
   усі точки на різних вузлах: добре
матриця за 0.097 с
недосяжних пар    : 0
асиметрія         : 0 м
коефіцієнт обходу : 1.16


> **Асиметрія.** $D_{ij} \\neq D_{ji}$ — не помилка: односторонній рух робить
> шлях туди і назад різним. На пішохідній мережі асиметрія мала, на
> автомобільній — помітна. Подивіться на своє число і поясніть його у звіті.

---
## 6. Задача комівояжера

| метод | що робить | складність |
|---|---|---|
| найближчий сусід | щоразу йде в найближчу невідвідану точку | $O(n^2)$ |
| 2-opt | розвертає ділянки готового маршруту, поки коротшає | $O(n^2)$ за прохід |
| Held–Karp | динамічне програмування за підмножинами — **точний** | $O(2^n n^2)$ часу, $O(2^n n)$ пам'яті |

Held–Karp дано готовим: він не є метою роботи, але без нього ви не дізнаєтесь,
**наскільки** помиляються перші два.

### Завдання 6.1 — реалізуйте перші два методи

In [ ]:
def tour_length(D, tour):
    return float(sum(D[tour[i], tour[i + 1]] for i in range(len(tour) - 1)))

def nearest_neighbour(D):
    k = len(D)
    tour, left = [0], set(range(1, k))
    while left:
        now = tour[-1]
        nxt = min(left, key=lambda j: D[now, j])
        tour.append(nxt); left.remove(nxt)
    return tour + [0]

def two_opt(D, tour):
    best, best_len = tour[:], tour_length(D, tour)
    improved = True
    while improved:
        improved = False
        for i in range(1, len(best) - 2):
            for j in range(i + 1, len(best) - 1):
                new = best[:i] + best[i:j + 1][::-1] + best[j + 1:]
                L = tour_length(D, new)
                if L < best_len - 1e-9:
                    best, best_len, improved = new, L, True
    return best

In [22]:
def held_karp(D):
    """Точний розв'язок. Вузол 0 — старт і фініш. ДАНО, писати не треба."""
    k = len(D); N = k - 1; INF = float('inf')
    dp = [[INF] * N for _ in range(1 << N)]
    par = [[-1] * N for _ in range(1 << N)]
    for j in range(N):
        dp[1 << j][j] = D[0, j + 1]
    for S in range(1 << N):
        for j in range(N):
            cur = dp[S][j]
            if cur == INF or not (S >> j) & 1:
                continue
            for t in range(N):
                if (S >> t) & 1:
                    continue
                S2 = S | (1 << t); v = cur + D[j + 1, t + 1]
                if v < dp[S2][t]:
                    dp[S2][t] = v; par[S2][t] = j
    full = (1 << N) - 1
    bj = min(range(N), key=lambda j: dp[full][j] + D[j + 1, 0])
    best = dp[full][bj] + D[bj + 1, 0]
    path = []; S = full; j = bj
    while j != -1:
        path.append(j + 1); pj = par[S][j]; S ^= (1 << j); j = pj
    return [0] + path[::-1] + [0], float(best)

### Самоперевірка

Ці твердження мають виконатись для **будь-якого** міста. Якщо хоч одне впало —
шукайте помилку в коді, а не в даних.

In [23]:
def check_tour(tour, k, label):
    assert tour[0] == 0 and tour[-1] == 0, '%s: маршрут має починатись і закінчуватись на старті' % label
    assert sorted(tour[:-1]) == list(range(k)), '%s: кожна точка рівно один раз' % label
    print('%s: структура маршруту правильна (%d точок)' % (label, k))

nn10 = nearest_neighbour(D10)
op10 = two_opt(D10, nn10)
ex10, L_opt = held_karp(D10)

for t, lbl in [(nn10, 'найближчий сусід'), (op10, '2-opt'), (ex10, 'Held-Karp')]:
    check_tour(t, len(ids10), lbl)

assert tour_length(D10, op10) <= tour_length(D10, nn10) + 1e-6, '2-opt не може бути гіршим за вхідний маршрут'
assert L_opt <= tour_length(D10, op10) + 1e-6, 'точний розв\u2019язок не може бути гіршим за наближений'
print()
print('усі перевірки пройдено')

найближчий сусід: структура маршруту правильна (11 точок)
2-opt: структура маршруту правильна (11 точок)
Held-Karp: структура маршруту правильна (11 точок)

усі перевірки пройдено


In [24]:
rows = []
for name, tour in [('найближчий сусід', nn10), ('2-opt', op10), ('Held-Karp (точний)', ex10)]:
    L = tour_length(D10, tour)
    rows.append({'метод': name, 'довжина, м': round(L),
                 'гірше оптимуму, %': round(100 * (L / L_opt - 1), 2)})
display(pd.DataFrame(rows).set_index('метод'))

print()
print('оптимальний порядок:')
for step, i in enumerate(ex10):
    print('  %2d. %s' % (step, pts10['name'].iloc[i]))
print()
print('довжина: %.2f км' % (L_opt / 1000))

,"довжина, м","гірше оптимуму, %"
метод,,
найближчий сусід,12147,6.68
2-opt,11387,0.00
Held-Karp (точний),11387,0.00



оптимальний порядок:
   0. СТАРТ: Рівне
   1. Марії Несвицькій (Рівненській)
   2. ЗіС-3
   3. Тарасу Шевченку
   4. Загиблим у війні 1941-1945 рр.
   5. Меморіал Слави
   6. Климу Савуру
   7. Володимир Мисечко
   8. Симону Петлюрі
   9. Жертвам Чорнобильскої катастрофи
  10. Літературний музей Уласа Самчука
  11. СТАРТ: Рівне

довжина: 11.39 км


### Завдання 6.2 — велика задача

Спершу оцініть, чи можливий тут точний розв'язок. Порахуйте для
$n = 11, 16, 21, 26, 31$ розмір таблиці Held–Karp ($2^{n-1}\\cdot(n-1)$ клітин)
і скільки це гігабайтів при 8 байтах на клітину.

In [25]:
rows = []
for n in [11, 16, 21, 26, 31]:
    cells = 2 ** (n - 1) * (n - 1)               # розмір таблиці Held-Karp
    rows.append({'n': n, 'клітин': cells, 'ГБ (8 байт/клітину)': round(cells * 8 / 1e9, 4)})
display(pd.DataFrame(rows).set_index('n'))

,клітин,ГБ (8 байт/клітину)
n,,
11,10240,0.0001
16,491520,0.0039
21,20971520,0.1678
26,838860800,6.7109
31,32212254720,257.6980


Подивіться на останній рядок і напишіть у звіті, **чому саме** точний розв'язок
на 30 точках недоступний. Слово «довго» відповіддю не є.

In [26]:
pts30 = make_points(top_score)
ids30, dist30 = snap(pts30)
check_snap(pts30, ids30, dist30, 'велика задача')

D30 = road_matrix(ids30)
E30 = straight_matrix(pts30)
print('коефіцієнт обходу: %.2f' % detour_factor(D30, E30))

nn30 = nearest_neighbour(D30)
t0 = time.time(); op30 = two_opt(D30, nn30); t_op30 = time.time() - t0
check_tour(op30, len(ids30), '2-opt на 30 точках')

L_nn30, L_op30 = tour_length(D30, nn30), tour_length(D30, op30)
print()
print('найближчий сусід : %8.0f м' % L_nn30)
print('2-opt            : %8.0f м  за %.3f с  (коротше на %.1f%%)'
      % (L_op30, t_op30, 100 * (1 - L_op30 / L_nn30)))
print()
print('ЦЕ — база порівняння для лабораторної роботи № 8.')

велика задача: точок 31, різних вузлів 31
   відстань точка -> вузол: медіана 17.9 м, максимум 50.6 м
   усі точки на різних вузлах: добре
коефіцієнт обходу: 1.21
2-opt на 30 точках: структура маршруту правильна (31 точок)

найближчий сусід :    25859 м
2-opt            :    21574 м  за 0.002 с  (коротше на 16.6%)

ЦЕ — база порівняння для лабораторної роботи № 8.


---
## 7. Просторова індексація: H3

H3 — ієрархічна сітка шестикутників. Рівень 9 — приблизно 200 м по ребру.

📖 <https://h3geo.org/docs/core-library/restable/>

### Завдання 7.1

Розкладіть **усіх** кандидатів по комірках H3 рівня 9 і побудуйте картограму
щільності. На тій самій карті покажіть межу міста і точки вашого маршруту.

In [ ]:
import h3
from shapely.geometry import Polygon
import branca.colormap as bcm

H3_RES = 9
print('середнє ребро комірки: %.0f м' % h3.average_hexagon_edge_length(H3_RES, unit='m'))

rp = cand.geometry.representative_point()
cand['h3'] = [h3.latlng_to_cell(p.y, p.x, H3_RES) for p in rp]

dens = cand['h3'].value_counts()
hexes = gpd.GeoDataFrame(
    {'h3': dens.index, 'n': dens.values},
    geometry=[Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in dens.index],
    crs='EPSG:4326')

cmap = bcm.linear.YlOrRd_09.scale(1, max(2, int(dens.max())))
pts_h3 = make_points(top10)
c0 = boundary.geometry.iloc[0].centroid
fmap_h3 = folium.Map(location=[c0.y, c0.x], zoom_start=13, tiles=BASEMAP)
folium.GeoJson(boundary.geometry.iloc[0].__geo_interface__,
               style_function=lambda f: {'fill': False, 'color': '#333', 'weight': 2}).add_to(fmap_h3)
folium.GeoJson(hexes.to_json(),
               style_function=lambda f: {'fillColor': cmap(f['properties']['n']), 'color': '#999',
                                         'weight': 0.5, 'fillOpacity': 0.65},
               tooltip=folium.GeoJsonTooltip(fields=['n'], aliases=["об'єктів:"])).add_to(fmap_h3)
for _, r in pts_h3.iterrows():
    folium.CircleMarker([r.geometry.y, r.geometry.x], radius=5, color='#0b3d91',
                        fill=True, fill_opacity=1, tooltip=r['name']).add_to(fmap_h3)
fmap_h3.save('lr5_h3.html')

cum = dens.sort_values(ascending=False).cumsum()
k_half = int((cum < len(cand) / 2).sum() + 1)
print('кандидатів: %d, комірок з ними: %d' % (len(cand), len(dens)))
print("половина пам'яток (%d) зосереджена у %d комірках із %d" % (len(cand) / 2, k_half, len(dens)))
print("найщільніша комірка: %d об'єктів" % dens.max())
fmap_h3

середнє ребро комірки: 201 м
кандидатів: 105, комірок з ними: 43
половина пам'яток (52) зосереджена у 8 комірках із 43
найщільніша комірка: 16 об'єктів


> **Пастка.** `h3.cell_to_boundary` повертає пари **(lat, lng)**, а
> `shapely.Polygon` чекає **(x, y)**, тобто (lng, lat). Переплутаєте — місто
> опиниться в Індійському океані. Це видно одразу, тому дивіться на карту.

### Питання до звіту

У скількох комірках зосереджено половину всіх пам'яток вашого міста?
Що це говорить про форму маршруту?

---
## 8. Просторове поєднання

### Завдання 8.1

Для кожної точки маршруту знайдіть найближчий заклад із шару `amenities`
(`sjoin_nearest`) і виведіть таблицю з відстанню в метрах.

Нагадування: `sjoin_nearest` рахує в одиницях CRS. У градусах результат буде
безглуздим — перепроєктуйте обидва шари в метричну систему.

In [ ]:
am = gpd.GeoDataFrame(
    {'заклад': amenities['name'], 'тип': amenities['amenity']},
    geometry=amenities.geometry.representative_point(), crs=amenities.crs).to_crs(32635)
am = am[am['заклад'].notna()]

pm = pts10.to_crs(32635)[['name', 'geometry']]

near = gpd.sjoin_nearest(pm, am, how='left', distance_col='відстань_м')
near = near[~near.index.duplicated(keep='first')]
near['відстань_м'] = near['відстань_м'].round(0)
display(near[['name', 'заклад', 'тип', 'відстань_м']].reset_index(drop=True))
print('медіана відстані до найближчого закладу: %.0f м' % near['відстань_м'].median())

,name,заклад,тип,відстань_м
0,СТАРТ: Рівне,Дорожнє,cafe,118.0
1,Загиблим у війні 1941-1945 рр.,Фламінго,restaurant,55.0
2,ЗіС-3,DarkSide2,cafe,42.0
3,Климу Савуру,У Тьоті Рози,cafe,35.0
4,Марії Несвицькій (Рівненській),Автомат питної води,drinking_water,36.0
5,Симону Петлюрі,"Мила, це я",cafe,42.0
6,Тарасу Шевченку,Шахівка,cafe,48.0
7,Літературний музей Уласа Самчука,Celentano,restaurant,99.0
8,Меморіал Слави,Фуксик,cafe,812.0
9,Володимир Мисечко,Bazzikalo,cafe,73.0


медіана відстані до найближчого закладу: 48 м


---
## 9. Маршрут на карті

Лінія маршруту — це **геометрія ребер графа**, а не відрізки між пам'ятками.

### Завдання 9.1

Побудуйте карту, на якій **обидві** лінії: справжній маршрут по ребрах і
прямі відрізки між точками. Різниця має бути видно оком.

In [ ]:
node_xy = nodes.set_index('osmid')[['x', 'y']]

def route_coords(tour, ids):
    """Координати маршруту по ребрах графа у форматі (lat, lon) для folium."""
    coords = []
    for a, b in zip(tour[:-1], tour[1:]):
        try:
            path = nx.shortest_path(G, ids[a], ids[b], weight='length')
        except nx.NetworkXNoPath:
            path = [ids[a], ids[b]]
        for nd in path:
            coords.append((float(node_xy.loc[nd, 'y']), float(node_xy.loc[nd, 'x'])))
    return coords

def straight_coords(tour, pts):
    return [(float(pts.geometry.y.iloc[i]), float(pts.geometry.x.iloc[i])) for i in tour]

c0 = boundary.geometry.iloc[0].centroid
fmap = folium.Map(location=[c0.y, c0.x], zoom_start=14, tiles=BASEMAP)

g30 = folium.FeatureGroup(name='2-opt, велика задача', show=False)
folium.PolyLine(route_coords(op30, ids30), color='#1f77b4', weight=3, opacity=.8).add_to(g30)
g30.add_to(fmap)

g10 = folium.FeatureGroup(name='точний маршрут (по ребрах)')
folium.PolyLine(route_coords(ex10, ids10), color='#d62728', weight=5, opacity=.9,
                tooltip='по ребрах графа').add_to(g10)
g10.add_to(fmap)
gs = folium.FeatureGroup(name='прямі відрізки')
folium.PolyLine(straight_coords(ex10, pts10), color='#444', weight=2, dash_array='6 6',
                tooltip='прямі відрізки').add_to(gs)
gs.add_to(fmap)

for n, i in enumerate(ex10[:-1]):
    lat, lon = float(pts10.geometry.y.iloc[i]), float(pts10.geometry.x.iloc[i])
    html = ('<div style="background:#d62728;color:#fff;border-radius:50%%;width:22px;height:22px;'
            'text-align:center;line-height:22px;font:bold 12px sans-serif;">%d</div>' % n)
    folium.Marker([lat, lon], icon=folium.DivIcon(html=html, icon_size=(22, 22), icon_anchor=(11, 11)),
                  tooltip='%d. %s' % (n, pts10['name'].iloc[i])).add_to(fmap)
folium.LayerControl().add_to(fmap)
fmap.save('lr5_маршрут.html')

print('довжина по ребрах: %.2f км; по прямих відрізках: %.2f км'
      % (tour_length(D10, ex10) / 1000, tour_length(E10, ex10) / 1000))
fmap

довжина по ребрах: 11.39 км; по прямих відрізках: 9.73 км


---
## 10. Результати для лабораторної роботи № 8

Збережіть файл, який ЛР8 прочитає і спробує побити.

In [30]:
result = {
    'студент': STUDENT_NAME,
    'номер': LIST_NUMBER,
    'місто': CITY,
    'мала_задача': {
        'точок': len(ids10),
        'оптимум_м': round(L_opt, 1),
        'оптимальний_порядок': [str(pts10['name'].iloc[i]) for i in ex10],
        'найближчий_сусід_м': round(tour_length(D10, nn10), 1),
        'двоопт_м': round(tour_length(D10, op10), 1),
    },
    'велика_задача': {
        'точок': len(ids30),
        'найближчий_сусід_м': round(L_nn30, 1),
        'двоопт_м': round(L_op30, 1),
        'двоопт_с': round(t_op30, 4),
        'точки': [{'name': str(pts30['name'].iloc[i]),
                   'lat': float(pts30.geometry.y.iloc[i]),
                   'lon': float(pts30.geometry.x.iloc[i])} for i in range(len(pts30))],
        'матриця_віддалей_м': [[None if not np.isfinite(v) else round(float(v), 1)
                                for v in row] for row in D30],
    },
}
with open('lr5_результати.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=1)
print('збережено lr5_результати.json (%.1f КБ)' % (os.path.getsize('lr5_результати.json') / 1024))

збережено lr5_результати.json (15.5 КБ)


---
## 11. Звіт

Відповіді пишіть просто в наступній комірці, числами зі своїх обчислень.

1. **Скільки об'єктів ви відкинули** на кроці 2 і за яким правилом? Чи погодились
   би ви віддати цей фільтр замовнику без пояснень?
2. **Які сигнали ввійшли у ваш рейтинг і з якими вагами?** Чому саме так?
3. **Скільки об'єктів збіглося** у ваших двох рейтингах із десяти? Що це
   означає для слова «найкращі»?
4. **Який у вас коефіцієнт обходу** і чи типовий він? Якщо ні — що в місті
   його пояснює?
5. **На скільки відсотків** найближчий сусід гірший за точний розв'язок?
   Чи здивувало вас це число?
6. **Чому точний розв'язок на 30 точках недоступний?** Назвіть конкретну
   величину, яка цьому заважає.
7. **Що у ваших даних виявилось неправильним** — дублікати, пропущені назви,
   об'єкти не на місці? Як ви з цим вчинили?

### Декларація про використання штучного інтелекту

Зазначте, чи користувались ШІ-помічником, у яких саме місцях і як перевіряли
отриманий код. Використання дозволене; **неперевірене** використання — ні.

In [1]:
REPORT = '''
1. Відкинуто 14 із 119 об'єктів шару heritage (лишилось 105 кандидатів). Правило: об'єкт має мати name; усі 14 відкинутих — без назви. Дрібних знаків (boundary_stone, wayside_cross тощо) у Рівному в шарі немає, тож цей фільтр відкинув 0. Окремо, на етапі дублікатів, видалено ще 44 об'єкти, що стоять ближче 60 м до вищого в рейтингу. Без пояснення замовнику цей фільтр віддати не можна: безіменні об'єкти зникають, і він має знати, що це свідоме рішення.

2. Сигнали й ваги: наявність wikidata/wikipedia — 0,35 (незалежне від OSM визнання, найнадійніший сигнал); кількість перекладів назви (rank pct) — 0,30 (перекладають те, що шукають іноземні туристи); музей/галерея — 0,15 (об'єкт призначений для відвідування); historic заповнений — 0,10 (історична природа); площа (rank pct) — 0,10 (найслабший сигнал: великим буває й склад). Сума ваг дорівнює 1; ваги — експертна оцінка, а не результат навчання, тому їх можна оскаржити. Водночас важливо враховувати нюанс функції rank(pct=True): вона ділить місце об'єкта на їх загальну кількість, а однакові значення отримують середнє місце. Через це 94 об'єкти без перекладів (n_lang = 0) отримали ранг приблизно 0,45 (а не 0), тому кожен такий об'єкт заробляє «бали ні за що» (0,45 × 0,30 ≈ 0,14 бала), що штучно вирівнює загальні оцінки.

3. У двох рейтингах із десяти збіглося 3 об'єкти до очищення дублікатів і 2 після. Саме порівняння є методично «нечесним»: рейтинг за комплексним балом був очищений від дублікатів (за правилом 60 метрів), тоді як рейтинг за площею побудовано на сирових даних (через що, наприклад, «Євро-Арт» фігурує там двічі). Крім того, у рейтингу за площею меморіальні дошки успадковують площу цілої будівлі, на якій вони висять, тобто цей критерій міряє розміри будинків, а не самих пам'яток. Це пояснює різницю результатів: відповіді отримано з тих самих даних тим самим кодом, але за різними правилами. Важливий висновок щодо складу топ-10: до роботи функції dedupe у першу десятку насправді входили всі три музеї (#8 Літературний, #9 Бурштину, #10 Краєзнавчий). Проте функція dedupe прибрала їх як «дублікати», бо вони стояли занадто близько до інших об'єктів (наприклад, Музей бурштину — за 6,2 м від Літературного; Краєзнавчий — за 56 м від гармати ЗіС-3). Це яскраво демонструє критичне обмеження: правило «ближче 60 м = дублікат» є занадто грубим для щільної забудови центру міста і знищує унікальні сусідні об'єкти, через що у фінальному топ-10 лишився лише один музей.

4. Коефіцієнт обходу малої задачі 1,16, великої 1,21 (типово 1,2–1,6). Мала задача трохи нижча за типовий діапазон: центр Рівного компактний, а пішохідна мережа має багато коротких проходів між кварталами, тож дорога майже не довша за пряму. Асиметрія матриці дорівнює 0 м: нульова асиметрія отримується саме завдяки структурі пішохідного графа (osmnx для network_type='walk' за побудовою робить кожну вулицю двосторонньою), а не через специфіку міста[cite: 7]. Недосяжних пар немає, усі точки посіли різні вузли (медіана відстані точка→вузол 17 м, максимум 30 м у малій і 51 м у великій задачі). Маршрут по ребрах (11,39 км) на 17 % довший за ламану з прямих відрізків (9,73 км).

5. Найближчий сусід гірший за точний розв'язок на 6,68 % (12 147 м проти 11 387 м). 2-opt на десяти точках дав той самий оптимум (0,00 %), що мене здивувало. На великій задачі 2-opt коротший за найближчого сусіда на 16,6 % (21 574 м проти 25 859 м) і працює за 0,002 с; оптимум там невідомий, тож порівнювати можна лише методи між собою.

6. Точний розв'язок на 30 точках недоступний через пам'ять, а не через час: таблиця Held–Karp має 2^(n−1)·(n−1) клітин, для n = 31 це 2^30 · 30 ≈ 3,2·10^10 клітин, тобто ≈258 ГБ при 8 байтах на клітину (для n = 26 було б 6,7 ГБ). Кожні +5 точок збільшують таблицю приблизно у 38 разів, і таку таблицю не вмістить жоден ноутбук.

7. Проблеми в даних: 14 об'єктів без назви; 19 пар ближче 60 м у топ-30, серед них дубль «Євро-Арт» двічі на відстані 12,4 м, дві меморіальні дошки за 3,5 м одна від одної біля краєзнавчого музею (23–26 м), літературний музей і музей бурштину за 6,2 м, військова техніка (ЗіС-3, ІС-2, ІС-3, «VW T4 (уламки)») окремими точками поруч із меморіалом; орфографічні помилки в назвах («Чорнобильскої», «митлополиту»); лише 13 зі 105 об'єктів мають площу, бо більшість — точки. Дії: безіменні відкинуто, у кожній групі близьких об'єктів лишено вищий у рейтингу (видалено 44), помилки в назвах не виправлено, щоб не змінювати джерело мовчки; як слабкий сигнал площу в рейтингу лишено з вагою 0,10. Щодо просторового розподілу (H3): половина пам'яток (52 із 105) зосереджена лише у 8 комірках із 43 (найщільніша містить 16 об'єктів). Це свідчить про те, що основна частина маршруту пролягає компактним історичним центром, проте окремі віддалені об'єкти — наприклад, Меморіал Слави, де найближчий заклад розташований на відстані 812 м проти загальної медіани 48 м — суттєво подовжують загальну довжину шляху[cite: 7].


Декларація про використання ШІ:
Інструмент: Gemini (Google). Згенеровано: допомога з формулюванням висновків, перевірка логіки розрахунків та підказки щодо оптимізації коду. Нюанси роботи ШІ: початкові пропозиції щодо окремих етапів потребували коригування, яке виконано самостійно в процесі діагностики та тестування скриптів. Самостійно перевірено: коректність метричної CRS, правильність порядку координат, звірка з контрольними ознаками на карті та візуальна перевірка побудованих об'єктів.
'''
print(REPORT)


1. Відкинуто 14 із 119 об'єктів шару heritage (лишилось 105 кандидатів). Правило: об'єкт має мати name; усі 14 відкинутих — без назви. Дрібних знаків (boundary_stone, wayside_cross тощо) у Рівному в шарі немає, тож цей фільтр відкинув 0. Окремо, на етапі дублікатів, видалено ще 44 об'єкти, що стоять ближче 60 м до вищого в рейтингу. Без пояснення замовнику цей фільтр віддати не можна: безіменні об'єкти зникають, і він має знати, що це свідоме рішення.

2. Сигнали й ваги: наявність wikidata/wikipedia — 0,35 (незалежне від OSM визнання, найнадійніший сигнал); кількість перекладів назви (rank pct) — 0,30 (перекладають те, що шукають іноземні туристи); музей/галерея — 0,15 (об'єкт призначений для відвідування); historic заповнений — 0,10 (історична природа); площа (rank pct) — 0,10 (найслабший сигнал: великим буває й склад). Сума ваг дорівнює 1; ваги — експертна оцінка, а не результат навчання, тому їх можна оскаржити. Водночас важливо враховувати нюанс функції rank(pct=True): вона ділить